# Atividade — Gráficos e análise de transporte

Notebook organizado conforme as Partes 1 e 2 da atividade.  
**Observação:** para executar as leituras de Excel, coloque o arquivo fornecido pelo professor na mesma pasta do notebook e ajuste o nome, se necessário.

In [ ]:
# Se necessário, instale as bibliotecas:
%pip install matplotlib pandas seaborn openpyxl

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## Parte 1 — Gráficos de pizza

Os valores abaixo são os percentuais do Brasil apresentados na Tabela 13 (2022). O gráfico compara os meios de transporte por cor/raça.

In [ ]:
meios = [
    "A pé", "Bicicleta", "Motocicleta ou mototáxi",
    "Automóvel, táxi ou assemelhados", "Transporte coletivo", "Outros"
]

dados_brasil = pd.DataFrame({
    "Meio de Transporte": meios,
    "Branca": [19.4, 3.2, 9.9, 41.8, 25.2, 0.5],
    "Preta ou Parda": [24.8, 5.2, 14.2, 20.6, 34.6, 0.6],
    "Indígena": [37.5, 5.9, 15.4, 15.2, 22.4, 3.6]
})

cores = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99', '#c2c2f0', '#ffb3e6']
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, grupo in enumerate(["Branca", "Preta ou Parda", "Indígena"]):
    axes[i].pie(
        dados_brasil[grupo],
        labels=dados_brasil["Meio de Transporte"],
        autopct='%1.1f%%',
        startangle=180,
        colors=cores
    )
    axes[i].set_title(f"Cor/Raça: {grupo}", fontweight="bold")

plt.suptitle("Meios de transporte das mulheres — Brasil (2022)", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

## Parte 2 — Análise com `groupby` e `melt`

A leitura abaixo considera uma planilha com os dados por unidade geográfica. Se o nome do arquivo ou da aba for diferente, altere os valores em `arquivo` e `aba`. O código mostra as colunas para facilitar a conferência.

In [ ]:
arquivo = "Tabela_13_Meio_de_transporte.xlsx"
aba = "BRUM"

df = pd.read_excel(arquivo, sheet_name=aba)
df.head()

In [ ]:
# Confira os nomes das colunas disponíveis
print(df.columns.tolist())
print(df.head())

### Preparação dos dados

O código a seguir transforma uma tabela larga em formato longo com `melt`. Ele pressupõe que a planilha tenha colunas de localização e colunas de meios de transporte. Ajuste `col_local` e os nomes em `colunas_transporte` conforme o resultado da célula anterior.

In [ ]:
# Ajuste estes nomes conforme as colunas da sua planilha.
col_local = "Brasil, Grande Região, Unidade da Federação e Município"

# Colunas correspondentes aos meios de transporte, caso estejam presentes.
colunas_transporte = [
    "A pé", "Bicicleta", "Motocicleta ou Mototaxi",
    "Automóvel, taxi ou assemelhados", "Transporte coletivo", "Outros"
]

# Mantém somente as colunas que realmente existem na planilha.
colunas_transporte = [c for c in colunas_transporte if c in df.columns]

long = df.melt(
    id_vars=[col_local],
    value_vars=colunas_transporte,
    var_name="Meio de Transporte",
    value_name="Percentual"
)

long["Percentual"] = (
    long["Percentual"].astype(str).str.replace(",", ".", regex=False)
)
long["Percentual"] = pd.to_numeric(long["Percentual"], errors="coerce")
long.head()

### 1. Listar apenas os estados

Remove linhas do Brasil, das regiões e dos municípios, mantendo as unidades da Federação.

In [ ]:
estados = [
    "Rondônia", "Acre", "Amazonas", "Roraima", "Pará", "Amapá", "Tocantins",
    "Maranhão", "Piauí", "Ceará", "Rio Grande do Norte", "Paraíba",
    "Pernambuco", "Alagoas", "Sergipe", "Bahia", "Minas Gerais",
    "Espírito Santo", "Rio de Janeiro", "São Paulo", "Paraná",
    "Santa Catarina", "Rio Grande do Sul", "Mato Grosso do Sul",
    "Mato Grosso", "Goiás", "Distrito Federal"
]

df_estados = long[long[col_local].isin(estados)].copy()
df_estados.head()

### 2. Estado com maior e menor uso de transporte coletivo

O cálculo considera os valores de transporte coletivo disponíveis na planilha.

In [ ]:
coletivo = df_estados[
    df_estados["Meio de Transporte"].str.contains("coletivo", case=False, na=False)
].dropna(subset=["Percentual"])

if not coletivo.empty:
    por_estado = coletivo.groupby(col_local)["Percentual"].mean()
    print("Maior uso:", por_estado.idxmax(), "-", por_estado.max(), "%")
    print("Menor uso:", por_estado.idxmin(), "-", por_estado.min(), "%")
else:
    print("Não foi encontrada uma coluna de transporte coletivo.")

### 3. Cidade com maior uso de bicicleta

Seleciona municípios (linhas com UF entre parênteses) e encontra o maior percentual de bicicleta.

In [ ]:
bicicleta = long[
    long["Meio de Transporte"].str.contains("bicicleta", case=False, na=False)
].dropna(subset=["Percentual"])

municipios = bicicleta[
    bicicleta[col_local].astype(str).str.contains(r"\([A-Z]{2}\)", regex=True, na=False)
]

if not municipios.empty:
    linha = municipios.loc[municipios["Percentual"].idxmax()]
    print("Cidade com maior uso de bicicleta:", linha[col_local], "-", linha["Percentual"], "%")
else:
    print("Não foram encontrados municípios com dados de bicicleta.")

### 4. Cidade com maior uso de automóvel



In [ ]:
automovel = long[
    long["Meio de Transporte"].str.contains("automóvel|automovel", case=False, na=False)
].dropna(subset=["Percentual"])

municipios_carro = automovel[
    automovel[col_local].astype(str).str.contains(r"\([A-Z]{2}\)", regex=True, na=False)
]

if not municipios_carro.empty:
    linha = municipios_carro.loc[municipios_carro["Percentual"].idxmax()]
    print("Cidade com maior uso de automóvel:", linha[col_local], "-", linha["Percentual"], "%")
else:
    print("Não foram encontrados municípios com dados de automóvel.")